# 01 · Premissas: de símbolos a vetores

> **Disciplina:** Representações de Linguagem — *como transformar palavra em vetor, o que esse vetor carrega e o viés que vem junto* (BLIS 2S26).
> Este notebook faz parte de uma sequência (`00` → `13`). Código de apoio em `src/replang`; interface interativa em `app/` (`uv run replang app`).


## Por que este notebook vem antes dos artigos?
Todos os artigos da disciplina partem de três ideias que raramente são explicadas neles:

1. **Hipótese distribucional** (Harris, 1954; Firth, 1957): *o significado de uma palavra está nos contextos em que ela ocorre* — "you shall know a word by the company it keeps".
2. **Espaço vetorial**: representar cada palavra como um ponto em $\mathbb{R}^d$ e medir "proximidade de significado" por uma **distância/ângulo** (cosseno).
3. **Compressão**: de uma representação esparsa e enorme (|V| dimensões) para uma densa e pequena (50–1000 dimensões) que **generaliza**.

Vamos percorrer o caminho histórico: *one-hot* → *bag-of-words* → matriz de coocorrência → PMI → SVD (LSA/HAL). No final, veremos por que o word2vec (notebook 02) é "a mesma coisa, aprendida por gradiente" (Levy & Goldberg, 2014).

In [1]:
# Configuração comum a todos os notebooks ---------------------------------------------
import sys, warnings, os
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"  # JupyterLab renderiza offline; HTML carrega plotly.js via CDN (notebooks leves)
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 160)
from replang import PATHS, WordVectors, settings
FAST = settings.fast  # REPLANG_FAST=1 reduz épocas/tamanhos
print("raiz:", ROOT, "| fast:", FAST)

raiz: Y:\BLIS\linguagemBLIS2S26\linguagem-blis2s26 | fast: False


## 1. One-hot: a representação sem significado

Com um vocabulário de tamanho $|V|$, a palavra $w_i$ vira o vetor $e_i$ (1 na posição $i$, 0 no resto).
Problemas: (a) dimensão $|V|$ (centenas de milhares); (b) **todos os pares são ortogonais** — `rei` está tão longe de `rainha` quanto de `banana`.

In [2]:
from replang.data.corpora import toy_corpus
from replang.models.cooccurrence import build_vocab, one_hot
toy = toy_corpus()
words, index, freq = build_vocab(toy)
print(f"{len(toy)} sentenças, |V| = {len(words)}")
e_rei, e_rainha, e_banana = (one_hot(words, w) for w in ["rei", "rainha", "banana"])
cos = lambda u, v: float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))
print("cos(rei, rainha) =", cos(e_rei, e_rainha), "| cos(rei, banana) =", cos(e_rei, e_banana))

18 sentenças, |V| = 57


cos(rei, rainha) = 0.0 | cos(rei, banana) = 0.0


## 2. Bag-of-words e TF-IDF: vetores para *documentos*

O primeiro uso de vetores em PLN foi para documentos (recuperação de informação, Salton, anos 1970): um documento é a contagem de suas palavras.
Hartmann (2016) usa exatamente o **cosseno entre TF-IDF** como *baseline* para similaridade de sentenças no ASSIN (notebook 07).

In [3]:
from replang.models.cooccurrence import bag_of_words, tfidf
bow = bag_of_words(toy, words)
tf = tfidf(bow)
docs = [" ".join(s) for s in toy]
sim = tf @ tf.T / (np.linalg.norm(tf, axis=1)[:, None] * np.linalg.norm(tf, axis=1)[None, :] + 1e-9)
i = 0
print("Documento:", docs[i])
for j in np.argsort(-sim[i])[1:4]:
    print(f"  {sim[i, j]:.2f}  {docs[j]}")

Documento: o rei governa o reino com a rainha
  1.00  a rainha governa o reino com o rei
  0.37  o príncipe brinca com o cachorro e a princesa brinca com o gato
  0.30  o rei come carne e a rainha come peixe


## 3. Matriz de coocorrência palavra × contexto

Agora o objeto central: para cada palavra-alvo $w$ contamos quantas vezes cada palavra de contexto $c$ aparece numa **janela** de $\pm k$ posições.
A linha $M_{w\cdot}$ é a "assinatura distribucional" de $w$. Duas palavras com linhas parecidas aparecem em contextos parecidos.

A ponderação pela distância (`harmonic` = $1/d$, usada pelo GloVe; `linear`, usada implicitamente pelo word2vec ao sortear a janela efetiva) é um detalhe que importa nos artigos.

In [4]:
from replang.models.cooccurrence import cooccurrence_matrix, CountModel
from replang.viz import cooccurrence_heatmap
M = cooccurrence_matrix(toy, index, window=2, weighting="flat")
sel = ["rei", "rainha", "príncipe", "princesa", "homem", "mulher", "gato", "cachorro", "maçã", "banana", "paris", "lisboa"]
df = pd.DataFrame(M.toarray(), index=words, columns=words).loc[sel, sel]
cooccurrence_heatmap(df, "Coocorrências (janela ±2) — subconjunto do toy corpus")

In [5]:
# As linhas completas (todas as colunas) já aproximam palavras do mesmo "tipo":
Mrow = M.toarray()
def cos_row(a, b):
    return cos(Mrow[index[a]], Mrow[index[b]])
for a, b in [("rei", "rainha"), ("rei", "banana"), ("gato", "cachorro"), ("maçã", "banana"), ("paris", "lisboa"), ("paris", "gato")]:
    print(f"cos({a}, {b}) = {cos_row(a, b):.2f}")

cos(rei, rainha) = 0.47
cos(rei, banana) = 0.24
cos(gato, cachorro) = 0.80
cos(maçã, banana) = 0.85
cos(paris, lisboa) = 0.71
cos(paris, gato) = 0.00


## 4. PMI e PPMI: o que é "coocorrer mais que o acaso"?

Contagens brutas são dominadas por palavras frequentes (*o*, *a*, *e* coocorrem com tudo). A **informação mútua pontual** compara a coocorrência observada com a esperada se $w$ e $c$ fossem independentes:

$$\mathrm{PMI}(w,c) = \log \frac{P(w,c)}{P(w)\,P(c)}, \qquad \mathrm{PPMI}(w,c)=\max(0,\ \mathrm{PMI}(w,c)).$$

Duas variantes aparecem "escondidas" nos artigos:
* **suavização** $P(c)^{\alpha}$ com $\alpha=0{,}75$ (Levy, Goldberg & Dagan, 2015) — o mesmo expoente $3/4$ da distribuição de ruído do word2vec (Mikolov 2013b, §2.2);
* **deslocamento** $-\log k$: Levy & Goldberg (2014) provaram que o Skip-gram com $k$ amostras negativas fatora implicitamente a matriz $\mathrm{PMI}(w,c) - \log k$.

In [6]:
from replang.models.cooccurrence import ppmi
P = ppmi(M, alpha=1.0)
dfp = pd.DataFrame(P.toarray(), index=words, columns=words).loc[sel, sel]
cooccurrence_heatmap(dfp, "PPMI (α = 1)")

In [7]:
# Efeito do PPMI sobre palavras funcionais: 'o' e 'a' deixam de dominar as linhas
top_counts = pd.Series(Mrow[index["rei"]], index=words).sort_values(ascending=False).head(6)
top_ppmi = pd.Series(P.toarray()[index["rei"]], index=words).sort_values(ascending=False).head(6)
pd.DataFrame({"contagem": top_counts.round(0), "PPMI": top_ppmi.round(2)})

,contagem,PPMI
a,1.0,NaN
carne,NaN,1.6
com,1.0,NaN
come,1.0,NaN
do,NaN,2.3
e,1.0,NaN
filho,NaN,2.3
governa,NaN,1.6
o,5.0,1.2
um,NaN,2.3


## 5. SVD: comprimir para generalizar (LSA / HAL)

A matriz PPMI é esparsa e tem |V| colunas. A **decomposição em valores singulares** $M \approx U_k \Sigma_k V_k^\top$ projeta cada palavra em $k$ dimensões densas, mantendo as direções de maior variância. É a *Latent Semantic Analysis* (Deerwester et al., 1990).
Usamos $U_k \Sigma_k^{1/2}$ como vetores — escolha empiricamente melhor (Levy et al., 2015) do que $U_k\Sigma_k$.

A classe `CountModel` encadeia contagem → PPMI → SVD e devolve um `WordVectors`, a mesma interface que usaremos para word2vec, GloVe e fastText.

In [8]:
from replang.viz import scatter_words
cm = CountModel(window=2, dim=6, alpha=0.75).fit(toy)
wv_count = cm.to_wordvectors("contagem→PPMI→SVD (toy)")
print("valores singulares:", np.round(cm.sigma_, 2))
for w in ["rei", "gato", "maçã", "paris"]:
    print(f"{w:<8}", [f"{x} ({s:.2f})" for x, s in wv_count.most_similar(w, topn=3)])
scatter_words(wv_count, {"realeza": ["rei", "rainha", "príncipe", "princesa"], "pessoas": ["homem", "mulher", "menino", "menina"],
                         "animais": ["gato", "cachorro"], "frutas": ["maçã", "banana"], "lugares": ["paris", "lisboa", "roma", "madri", "frança", "portugal", "itália", "espanha"]},
              title="Vetores densos (6d) do toy corpus, projetados por PCA")

valores singulares: [11.54  9.76  9.18  9.06  8.64  7.41]
rei      ['príncipe (0.99)', 'governa (0.99)', 'brinca (0.98)']
gato     ['cachorro (0.99)', 'dorme (0.92)', 'late (0.90)']
maçã     ['a (0.97)', 'lisboa (0.91)', 'frança (0.91)']
paris    ['roma (1.00)', 'lisboa (0.99)', 'madri (0.90)']


## 6. O mesmo pipeline num corpus real (Machado de Assis)

Com 2,4 M tokens, a matriz PPMI tem dezenas de milhares de linhas; a SVD truncada esparsa (`scipy.sparse.linalg.svds`) resolve em segundos para $k = 100$.
Compare os vizinhos com os do Wikipedia2Vec (treinado em 200 M+ tokens com Skip-gram).

In [9]:
import time
from replang.data.corpora import load_machado_sentences
sents = load_machado_sentences(40000 if FAST else None)
t = time.time()
cm_m = CountModel(window=4, dim=100, min_count=10, alpha=0.75, weighting="harmonic", max_vocab=15000).fit(sents)
wv_m = cm_m.to_wordvectors("PPMI-SVD 100d (Machado)")
print(f"{wv_m} em {time.time()-t:.0f}s; nnz(PPMI) = {cm_m.ppmi_.nnz:,}")
from replang.data.embeddings import load_ptwiki
pt = load_ptwiki()
for w in ["capitu", "amor", "dinheiro", "rua", "triste"]:
    print(f"{w:<9} PPMI-SVD: {[x for x, _ in wv_m.most_similar(w, topn=5)]}")
    print(f"{'':<9} Wiki2Vec: {[x for x, _ in pt.most_similar(w, topn=5)] if w in pt else '(fora do vocabulário da Wikipédia: personagem de Machado!)'}")

WordVectors(name='PPMI-SVD 100d (Machado)', n=14312, dim=100) em 10s; nnz(PPMI) = 2,554,851
capitu    PPMI-SVD: ['virgília', 'escobar', 'flora', 'sofia', 'carmo']
          Wiki2Vec: (fora do vocabulário da Wikipédia: personagem de Machado!)
amor      PPMI-SVD: ['coração', 'sentimento', 'afeto', 'orgulho', 'casto']
          Wiki2Vec: ['amar', 'encanto', 'ciúme', 'felicidade', 'ódio']
dinheiro  PPMI-SVD: ['emprego', 'preço', 'pagar', 'remédio', 'lhes']
          Wiki2Vec: ['subornos', 'pagar', 'suborno', 'quantias', 'propina']
rua       PPMI-SVD: ['esquina', 'ouvidor', 'quitanda', 'direção', 'praia']
          Wiki2Vec: ['avenida', 'esquina', 'praça', 'alameda', 'av']
triste    PPMI-SVD: ['alegre', 'fiquei', 'aborrecido', 'grave', 'tranqüilo']
          Wiki2Vec: ['infeliz', 'feliz', 'aborrecido', 'tristeza', 'horrível']


## 7. Cosseno, produto interno e normalização

Com vetores de norma 1, $\cos(u,v) = u \cdot v$. Bolukbasi et al. (2016, §3) assumem exatamente isso ("as is common, similarity is measured by inner product; words are normalized to unit length").
A norma de um vetor word2vec cresce com a frequência da palavra — por isso normalizar é importante antes de comparar.

In [10]:
norms = np.linalg.norm(pt.vectors, axis=1)
df = pd.DataFrame({"palavra": pt.words, "posição (freq. decrescente)": np.arange(len(pt)), "norma": norms})
import plotly.express as px
fig = px.scatter(df.iloc[::25], x="posição (freq. decrescente)", y="norma", hover_name="palavra", opacity=0.5,
                 title="Norma do vetor × ranking de frequência (Wikipedia2Vec PT)", log_x=True, template="plotly_white")
fig.show()
u, v = pt["rei"], pt["rainha"]
print("produto interno bruto:", float(u @ v), "| cosseno:", pt.similarity("rei", "rainha"))

produto interno bruto: 9.956557273864746 | cosseno: 0.731939971446991


## 8. Síntese: o que levar para o notebook 02

* **Significado ≈ distribuição de contextos.** Toda representação distribucional, de LSA a BERT, formaliza isso de um jeito.
* **Contagem + reponderação (PPMI) + compressão (SVD)** já produz vizinhos razoáveis. Baroni et al. (2014) mostraram que modelos *preditivos* (word2vec) ganham na maioria dos benchmarks, mas Levy et al. (2015) mostraram que boa parte da diferença são **hiperparâmetros** (janela, suavização, deslocamento, normalização).
* O word2vec aprende, por gradiente estocástico e sem construir a matriz, uma fatoração parecida — com a vantagem de escalar para bilhões de tokens em um dia (Mikolov 2013b, §1).

### Perguntas para discussão
1. Que informação a janela ±2 captura que a janela ±10 não captura, e vice-versa? (sintagmático × paradigmático; tópico × função)
2. O PPMI zera associações *negativas*. O que se perde? (Levy et al. discutem o "shifted PPMI".)
3. Se a hipótese distribucional é a premissa, o que acontece com palavras que **nunca** aparecem no corpus? (gancho para fastText, notebook 05)

## Referências
- HARRIS, Z. **Distributional Structure.** Word, 1954; FIRTH, J. R. *A synopsis of linguistic theory*, 1957.
- LEVY, O.; GOLDBERG, Y. **Neural Word Embedding as Implicit Matrix Factorization.** NIPS, 2014; LEVY, GOLDBERG & DAGAN. **Improving Distributional Similarity with Lessons Learned from Word Embeddings.** TACL, 2015.
- BARONI, M.; DINU, G.; KRUSZEWSKI, G. **Don't count, predict!** ACL, 2014.
- MIKOLOV, T.; CHEN, K.; CORRADO, G.; DEAN, J. **Efficient Estimation of Word Representations in Vector Space.** ICLR Workshop, 2013. [arXiv:1301.3781](https://arxiv.org/abs/1301.3781)
- PENNINGTON, J.; SOCHER, R.; MANNING, C. **GloVe: Global Vectors for Word Representation.** EMNLP, 2014. [ACL D14-1162](https://aclanthology.org/D14-1162/)